# Notebook 3: Measurement and the Born Rule

We have seen that a quantum state is a vector of complex amplitudes, and the Born rule
tells us that $P(i) = |\alpha_i|^2$. But what actually happens when we **measure**?

This notebook covers:

1. **Single measurement with collapse** -- after measurement, the state jumps to a basis state.
2. **Post-measurement stability** -- re-measuring a collapsed state always gives the same outcome.
3. **Repeated measurements** -- many independent measurements on copies of the same state
   recover the Born probabilities.
4. **Histogram visualization** -- seeing the Born rule in action.

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_demo.states import qubit_state, amplitudes_to_probabilities
from quantum_demo.measurement import measure_state, repeated_measurements
from quantum_demo.viz.static_plots import plot_probabilities

## 1. A single measurement

Consider the state $|\psi\rangle = \frac{1}{\sqrt{5}}|0\rangle + \frac{2}{\sqrt{5}}|1\rangle$,
which has Born probabilities $P(0) = 0.2$ and $P(1) = 0.8$.

When we measure, **one** of the basis states is observed, chosen randomly according
to the Born probabilities. The key quantum feature is **collapse**: after measurement,
the state is no longer a superposition -- it becomes the basis state that was observed.

In [ ]:
# Prepare the state
psi = qubit_state(1, 2)  # normalized to (1/sqrt(5), 2/sqrt(5))
print("State before measurement:", psi)
print("Born probabilities:", amplitudes_to_probabilities(psi))

In [ ]:
# Perform a single measurement (seeded for reproducibility)
rng = np.random.default_rng(seed=42)

outcome, collapsed, probs = measure_state(psi, rng=rng)

print(f"Observed outcome: {outcome}")
print(f"Collapsed state:  {collapsed}")
print(f"Pre-measurement probabilities: {probs}")

Notice that the collapsed state is a **basis vector** -- all amplitude is concentrated on
the observed outcome. The superposition has been destroyed.

## 2. Post-measurement stability

A fundamental feature of quantum measurement: if you immediately measure the collapsed
state again, you get the **same outcome** with certainty.

This is because the collapsed state is a basis vector, and basis vectors have all
their probability on a single outcome.

In [ ]:
# Measure the collapsed state several times
print(f"Collapsed state after first measurement: {collapsed}")
print(f"Probabilities of collapsed state: {amplitudes_to_probabilities(collapsed)}")
print()

rng2 = np.random.default_rng(seed=99)
print("Re-measuring the collapsed state 10 times:")
for i in range(10):
    outcome2, collapsed2, _ = measure_state(collapsed, rng=rng2)
    print(f"  Measurement {i+1}: outcome = {outcome2}")

Every re-measurement gives the same result. The collapse is **irreversible** (at least
within standard quantum mechanics). Once the superposition is gone, you cannot recover
the original amplitudes by measuring again.

This is why quantum algorithms must carefully orchestrate gates *before* measurement --
measurement is the final, destructive readout.

## 3. Repeated measurements: recovering the Born probabilities

A single measurement gives one random outcome. But if we prepare **many identical copies**
of $|\psi\rangle$ and measure each one, the histogram of outcomes converges to the
Born probabilities.

This is the operational meaning of $P(i) = |\alpha_i|^2$: it predicts the **long-run
frequency** of each outcome.

In [ ]:
# Prepare the state again
psi = qubit_state(1, 2)
print("State:", psi)
print("Born probabilities:", amplitudes_to_probabilities(psi))

In [ ]:
# Run repeated measurements with different shot counts
rng = np.random.default_rng(seed=123)

for shots in [10, 100, 1000, 10000]:
    counts = repeated_measurements(psi, shots=shots, rng=rng)
    freq_0 = counts.get(0, 0) / shots
    freq_1 = counts.get(1, 0) / shots
    print(f"Shots={shots:>5d}:  P(0)={freq_0:.4f}  P(1)={freq_1:.4f}  "
          f"(theory: 0.2000, 0.8000)")

As the number of shots increases, the empirical frequencies approach the theoretical
Born probabilities. This convergence is a direct quantum analogue of the classical
law of large numbers.

## 4. Histogram visualization

Let's make a clean visual comparison of the theoretical probabilities and the empirical
histogram from 10,000 shots.

In [ ]:
# Large-scale measurement
rng = np.random.default_rng(seed=456)
shots = 10_000
counts = repeated_measurements(psi, shots=shots, rng=rng)

# Build arrays for plotting
labels = ['|0>', '|1>']
theoretical = amplitudes_to_probabilities(psi)
empirical = np.array([counts.get(i, 0) / shots for i in range(2)])

print(f"Theoretical: {theoretical}")
print(f"Empirical ({shots:,} shots): {empirical}")

In [ ]:
# Side-by-side bar chart
x = np.arange(2)
width = 0.35

fig, ax = plt.subplots(figsize=(6, 4))
bars1 = ax.bar(x - width/2, theoretical, width, label='Born Rule (theory)',
               color='#4C72B0', edgecolor='black')
bars2 = ax.bar(x + width/2, empirical, width, label=f'Empirical ({shots:,} shots)',
               color='#55A868', edgecolor='black')

ax.set_ylim(0, 1.0)
ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title('Born Rule vs Empirical Measurement Frequencies')
ax.legend()
fig.tight_layout()
plt.show()

## 5. A richer example: equal superposition

Let's try the same exercise with $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$,
which should give a perfect 50/50 split.

In [ ]:
plus = qubit_state(1, 1)
print("State |+>:", plus)
print("Born probabilities:", amplitudes_to_probabilities(plus))

rng = np.random.default_rng(seed=789)
counts_plus = repeated_measurements(plus, shots=10_000, rng=rng)

empirical_plus = np.array([counts_plus.get(i, 0) / 10_000 for i in range(2)])
print(f"Empirical (10,000 shots): {empirical_plus}")

In [ ]:
# Convergence plot: track running frequency as a function of number of shots
rng = np.random.default_rng(seed=321)
all_shots = 5000
outcomes = []
for _ in range(all_shots):
    idx, _, _ = measure_state(plus, rng=rng)
    outcomes.append(idx)

outcomes = np.array(outcomes)
running_freq_0 = np.cumsum(outcomes == 0) / np.arange(1, all_shots + 1)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(running_freq_0, color='#4C72B0', linewidth=0.8)
ax.axhline(0.5, color='#C44E52', linestyle='--', linewidth=1.5, label='Theory: P(0) = 0.5')
ax.set_xlabel('Number of measurements')
ax.set_ylabel('Running frequency of |0>')
ax.set_title('Convergence to Born probability')
ax.set_ylim(0.3, 0.7)
ax.legend()
fig.tight_layout()
plt.show()

The running frequency oscillates at first but steadily converges toward the theoretical
value of 0.5. This is how the Born rule manifests experimentally.

## 6. Key takeaways

| Concept | Description |
|---------|-------------|
| **Born rule** | $P(i) = |\alpha_i|^2$ -- amplitudes squared give probabilities |
| **Collapse** | After measurement, the state becomes the observed basis vector |
| **Stability** | Re-measuring a collapsed state always gives the same result |
| **Statistics** | Repeated measurements on identical copies recover Born probabilities |
| **Irreversibility** | Measurement destroys superposition -- the original amplitudes are lost |

In the next notebook, we will see the phenomenon that makes quantum mechanics truly
different from classical probability: **interference** -- the ability of amplitudes
to add constructively or destructively before being squared.